# Dataset Preparation

In [2]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import duckdb

from pathlib import Path

In [3]:
# Pathing
project_root = Path.cwd()

if project_root.name == 'notebooks':
    project_root = project_root.parent

data_dir = project_root/'data'/'raw'/'m5'

sales_path = data_dir/'sales_train_evaluation.csv'
calendar_path = data_dir/'calendar.csv'
prices_path = data_dir/'sell_prices.csv'

print(f'Project root: {project_root}')
print(f'Data directory: {data_dir}')

Project root: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting
Data directory: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\data\raw\m5


In [4]:
# Df variable assignments
sales_df = pd.read_csv(sales_path)
calendar_df = pd.read_csv(calendar_path)
prices_df = pd.read_csv(prices_path)

In [5]:
# Daily observations
day_columns = [column for column in sales_df.columns
    if column.startswith('d_') # indicating the day of the sale
]

In [6]:
# Selecting store CA_1 as the location of interest
store_id = 'CA_1'

store_sales = sales_df.loc[sales_df['store_id'] == store_id].copy()

In [7]:
# representative development e.g.
item_summary = store_sales[['item_id', 'dept_id', 'cat_id']].copy()

item_summary['total_sales'] = (store_sales[day_columns].sum(axis=1))

item_summary['zero_rate'] = (
    store_sales[day_columns]
    .eq(0)
    .mean(axis=1)
)

In [8]:
# Dividing the products into three intermittency
item_summary['demand_group'] = pd.qcut(
    item_summary['zero_rate'],
    q=3,
    labels=[
        'low_intermittency',
        'medium_intermittency',
        'high_intermittency'
    ]
)

In [9]:
# Sample products from each groups
selected_item_ids = []

for _, group in item_summary.groupby(['cat_id', 'demand_group'], observed=True):
    sample = group.sample(n=min(12, len(group)), random_state=10)

    selected_item_ids.extend( sample['item_id'].tolist())

print(f'Each group has: {len(selected_item_ids)}')

Each group has: 108


### Reshaping selected products

In [10]:
selected_sales = store_sales.loc[store_sales['item_id'].isin(selected_item_ids)]

sales_long = selected_sales.melt(
    id_vars=[
        'id',
        'item_id',
        'dept_id',
        'cat_id',
        'store_id',
        'state_id'
    ],
    value_vars=day_columns,
    var_name='d',
    value_name='sales'
)

In [11]:
# Adding calendar informations
calendar_columns = [
    'd',
    'date',
    'wm_yr_wk',
    'weekday',
    'wday',
    'month',
    'year',
    'event_name_1',
    'event_type_1',
    'event_name_2',
    'event_type_2'
]

model_df = sales_long.merge(
    calendar_df[calendar_columns],
    on='d',
    how='left',
    validate='many_to_one'
)

In [12]:
# Adding weekly prices
model_df = model_df.merge(
    prices_df[
        [
            'store_id',
            'item_id',
            'wm_yr_wk',
            'sell_price'
        ]
    ],
    on=[
        'store_id',
        'item_id',
        'wm_yr_wk'
    ],
    how='left',
    validate='many_to_one'
)

### Checking modeling dataset

In [13]:
model_df.shape

(209628, 19)

In [14]:
model_df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,sell_price
0,HOBBIES_1_006_CA_1_evaluation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,NaN
1,HOBBIES_1_008_CA_1_evaluation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,d_1,12,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,0.46
2,HOBBIES_1_061_CA_1_evaluation,HOBBIES_1_061,HOBBIES_1,HOBBIES,CA_1,CA,d_1,1,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,2.76
3,HOBBIES_1_072_CA_1_evaluation,HOBBIES_1_072,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,NaN
4,HOBBIES_1_076_CA_1_evaluation,HOBBIES_1_076,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,2.94


In [15]:
# Missingness
model_df[
    [
        'date',
        'item_id',
        'sales',
        'sell_price'
    ]].isna().sum()

date              0
item_id           0
sales             0
sell_price    41594
dtype: int64

In [16]:
# Checking for duplicates
model_df.duplicated(
    subset=[
        'item_id',
        'store_id',
        'date'
    ]).sum()

np.int64(0)

In [17]:
# sorting the model dataset by date and item_id
model_df['date'] = pd.to_datetime(
    model_df['date']
)

model_df = model_df.sort_values(
    ['item_id', 'date']
).reset_index(drop=True)

In [18]:
# Saving the joined dataset
model_df['date'] = pd.to_datetime(
    model_df['date']
)

model_df = model_df.sort_values(
    ['item_id', 'date']
).reset_index(drop=True)

In [20]:
output_path = (
    project_root /
    'data' /
    'interim' /
    'ca1_development_joined.parquet'
)

model_df.to_parquet(output_path, index=False)

print(f'Saved to: {output_path}')

Saved to: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\data\interim\ca1_development_joined.parquet


Feature engineering done to select the most powerful predictors.
Among others: lag_1, lag_7, lag_14, lag_28, rolling_mean_7, rolling_mean_28, rolling_std_7, weekday, month, event indicator, selling price, price change, item, department, and category identifiers